In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 📚 Academic Research Crew

## What We're Building

A 4-agent academic research assistant:

| Agent | Role |
|-------|------|
| **Literature Searcher** | Find relevant papers and sources |
| **Summarizer** | Create concise summaries of key findings |
| **Gap Finder** | Identify research gaps and opportunities |
| **Bibliography Formatter** | Format citations properly |

## Key CrewAI Concept: `markdown=True`

The final bibliography task uses `markdown=True` to ensure the output
is properly formatted with headers, lists, and citations.

## Stack
- **CrewAI** — multi-agent pipeline
- **Ollama** — local LLM

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(model="ollama/qwen3.5:9b", base_url="http://localhost:11434")
print(f"LLM ready: {llm.model}")

## Step 2 — Research Topic

In [ ]:
research_topic = """
Topic: Retrieval-Augmented Generation (RAG) for domain-specific applications
Field: Natural Language Processing / Information Retrieval
Focus: How RAG systems perform when applied to specialized domains
like legal, medical, and financial text — compared to general-purpose LLMs.
Time range: 2022-2025
"""

print("Research topic loaded")

## Step 3 — Create Agents

In [ ]:
lit_searcher = Agent(
    role="Literature Search Specialist",
    goal="Find and catalog the most relevant academic papers on the research topic",
    backstory=(
        "You are a research librarian with expertise in computer science literature. "
        "You know how to search across arXiv, Semantic Scholar, and Google Scholar. "
        "You prioritize papers by citation count, venue quality, and recency."
    ),
    llm=llm,
    verbose=True,
)

summarizer = Agent(
    role="Research Paper Summarizer",
    goal="Create concise, accurate summaries of academic papers",
    backstory=(
        "You are a PhD researcher who writes literature review sections for top-tier "
        "conferences. You extract the key contribution, methodology, results, and "
        "limitations from each paper. You never misrepresent findings."
    ),
    llm=llm,
    verbose=True,
)

gap_finder = Agent(
    role="Research Gap Analyst",
    goal="Identify unexplored areas and promising research directions",
    backstory=(
        "You are a senior professor who mentors PhD students. You excel at reading "
        "across papers to find what's MISSING — the questions nobody has asked, "
        "the methods nobody has tried, the datasets nobody has created."
    ),
    llm=llm,
    verbose=True,
)

bib_formatter = Agent(
    role="Bibliography & Citation Formatter",
    goal="Compile and format a complete bibliography with proper citations",
    backstory=(
        "You are a meticulous academic editor who ensures every citation is correct. "
        "You format references in APA style and organize them thematically. "
        "You also create a reading order recommendation for newcomers to the field."
    ),
    llm=llm,
    verbose=True,
)

print("4 agents created")

## Step 4 — Create Tasks

In [ ]:
search_task = Task(
    description=f"""Find the most relevant academic papers for this research topic:

{research_topic}

Find 8-10 papers and for each provide:
1. Title, authors, year, venue (conference/journal)
2. Key contribution in one sentence
3. Why it's relevant to this research topic
4. Estimated citation count

Organize papers by sub-theme:
- Foundational RAG papers
- Domain-specific RAG applications
- Evaluation methodologies
- Recent advances (2024-2025)""",
    expected_output="A catalog of 8-10 relevant papers organized by theme.",
    agent=lit_searcher,
)

summary_task = Task(
    description="""For each paper found by the literature searcher, create a detailed summary:

For each paper:
1. PROBLEM: What problem does this paper solve?
2. METHOD: What approach/architecture do they use?
3. KEY RESULTS: Main findings with quantitative results if available
4. LIMITATIONS: What are the acknowledged or apparent limitations?
5. RELEVANCE: How does this connect to our research topic?

End with a synthesis: What do these papers collectively tell us?""",
    expected_output="Detailed summaries of each paper plus a synthesis.",
    agent=summarizer,
)

gap_task = Task(
    description="""Based on the literature review above, identify research gaps:

1. METHODOLOGICAL GAPS: What methods haven't been explored?
   - Are there architectures not yet applied to domain-specific RAG?
   - Are there evaluation metrics missing?

2. DOMAIN GAPS: Which domains are understudied?
   - Has RAG been tested in low-resource languages for specialized domains?
   - Are there industries where RAG hasn't been applied?

3. SCALABILITY GAPS: What about real-world deployment challenges?

4. Propose 3 concrete research questions that could fill these gaps.

5. For each question, suggest a feasible methodology.""",
    expected_output="Research gaps analysis with 3 proposed research questions.",
    agent=gap_finder,
)

bib_task = Task(
    description="""Compile the final research brief with formatted bibliography:

1. EXECUTIVE SUMMARY: 200-word overview of the research landscape
2. THEMATIC BIBLIOGRAPHY: Papers grouped by theme in APA format
3. KEY FINDINGS: Top 5 insights from the literature
4. RESEARCH GAPS: Summarized from the gap analysis
5. RECOMMENDED READING ORDER: For someone new to this field
6. SUGGESTED NEXT STEPS: For a researcher starting in this area""",
    expected_output="A complete research brief with formatted bibliography.",
    agent=bib_formatter,
    markdown=True,  # Ensures nice markdown formatting
)

print("4 tasks created")

## Step 5 — Run the Crew

In [ ]:
research_crew = Crew(
    agents=[lit_searcher, summarizer, gap_finder, bib_formatter],
    tasks=[search_task, summary_task, gap_task, bib_task],
    process=Process.sequential,
    verbose=True,
)

print("Academic research crew assembled!")
result = research_crew.kickoff()
print("\n✅ Research brief complete!")

In [ ]:
print("📚 FINAL RESEARCH BRIEF:")
print("=" * 60)
print(result.raw)

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **markdown=True** | Final task output formatted as proper markdown |
| **Thematic organization** | Papers grouped by sub-theme, not just listed |
| **Gap analysis** | Pro-active identification of what's missing |
| **Research questions** | Concrete, testable questions from the gaps |

## 🔧 Extensions

- **Semantic Scholar API**: Add real paper search with `crewai_tools`
- **PDF parsing**: Feed actual papers into agents with RAG tools
- **Citation graph**: Build a citation network visualization
- **output_file**: Save the bibliography to a .bib file